### **Project overview: Our chatbot's workflow**

Our chatbot will follow a four-step process to answer a user's question:

1. **Query Understanding Phase (Agent 1):**
The user’s question is first handled by a query understanding agent. This agent’s role is to interpret the user’s intent and extract key details from the question.

2. **Document Retrieval Phase (Agent 2):**
The interpreted query is used to search the restaurant’s FAQ PDF. This agent’s role is to retrieve only the most relevant sections of the document.

3. **Memory Phase (Agent 3):**
The retrieved information and user interaction are passed to a memory agent. This agent’s role is to store and recall conversation context, supporting more personalized responses.

4. **Response Generation Phase (Agent 4 – LLM):**
The retrieved FAQ content, along with memory context, is sent to an LLM-based response agent. This agent’s role is to generate a clear, friendly, and complete answer for the customer.

```
┌──────────────┐
│   User       │
│ (Question)   │
└──────┬───────┘
       │
       ▼
┌───────────────────────────┐
│ Query Understanding       │
│ Agent (Agent 1)           │
│ - Interprets user intent  │
│ - Extracts key keywords   │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ Document Retrieval        │
│ Agent (Agent 2)           │
│ - Searches FAQ PDF        │
│ - Retrieves relevant text │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ Memory Agent              │
│ (Agent 3)                 │
│ - Stores past questions   │
│ - Recalls conversation    │
│   context                 │
└──────┬────────────────────┘
       │
       ▼
┌───────────────────────────┐
│ LLM Response Agent        │
│ (Agent 4)                 │
│ - Combines retrieved info │
│ - Uses memory context     │
│ - Generates friendly reply│
└──────┬────────────────────┘
       │
       ▼
┌──────────────┐
│   User       │
│ (Final Reply)│
└──────────────┘
```



In [1]:
!pip install pypdf scikit-learn nltk


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install PyPDF2


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import re
import nltk
import numpy as np
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')

# Ensure the FAQ PDF is in the same directory or provide the full path
# You can also download it from the source provided in the example notebook
# 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/7vgNfis17dQfjHAiIKkBOg/The-Daily-Dish-FAQ.pdf'
faq_pdf_path = "The_Daily_Dish_FAQ.pdf"

[nltk_data] Downloading package punkt to /Users/treza/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
# Query agent


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
class DailyDishAgent:
    # Initialize the agent with a list of questions and corresponding answers
    def __init__(self, questions, answers):
        self.questions = questions  # List of sample or known questions
        self.answers = answers      # List of answers mapped to the questions
        # Initialize the TF-IDF vectorizer to convert text into numerical vectors
        self.vectorizer = TfidfVectorizer(
            stop_words="english",    # Remove common English stop words
            ngram_range=(1, 2),       # Use unigrams and bigrams for better context
            max_features=1000         # Limit vocabulary size to top 1000 features
        )
        # Convert all questions into TF-IDF vectors for similarity comparison
        self.doc_vectors = self.vectorizer.fit_transform(questions)

In [10]:
def answer(self, query):
    # Convert the user query into a TF-IDF vector
    # The query must be wrapped in a list as the vectorizer expects an iterable
    query_vector = self.vectorizer.transform([query])
    # Calculate cosine similarity between the query vector
    # and all stored question vectors
    similarities = cosine_similarity(
        query_vector,
        self.doc_vectors
    )[0]  # Extract similarity scores as a 1D array
    # Identify the index of the most similar question
    best_idx = similarities.argmax()
    # Retrieve the highest similarity score
    best_score = similarities[best_idx]
    # If the similarity score exceeds the predefined threshold,
    # return the corresponding answer
    if best_score >= 0.08:
        return self.answers[best_idx]
    else:
        # Fallback response when no good match is found
        return "I don't have information about that. Please contact us directly."

In [11]:
def find_best_match(self, query, threshold=0.08):
    # Convert the user query into a TF-IDF vector
    query_vector = self.vectorizer.transform([query])
    # Compute cosine similarity between the query vector
    # and all stored question vectors
    similarities = cosine_similarity(query_vector, self.doc_vectors)[0]
    # Identify the indices of the top 3 most similar questions
    top_indices = similarities.argsort()[-3:][::-1]
    # Retrieve the similarity scores for the top matches
    top_scores = similarities[top_indices]
    results = []  # List to store matching results
    # Iterate through the top matches and filter by similarity threshold
    for idx, score in zip(top_indices, top_scores):
        if score >= threshold:
            results.append({
                "question": self.questions[idx],  # Matched question text
                "answer": self.answers[idx],      # Corresponding answer
                "score": score                    # Similarity score
            })
    # Return the list of best matches (can be empty if no match meets the threshold)
    return results

In [12]:
# List of frequently asked customer questions
# These questions will be used as reference data for matching user queries
questions = [
    "What are your opening hours?",
    "Do you take reservations?",
    "What type of cuisine do you serve?",
    "Do you have vegetarian options?",
    "Where are you located?",
    "Do you offer delivery?",
    "What is your phone number?",
    "Do you have gluten-free options?"
]
# Corresponding answers to each question above
# The index of each answer matches the index of its related question
answers = [
    "We're open Monday-Thursday 11am-10pm, Friday-Saturday 11am-11pm, Sunday 10am-9pm.",
    "Yes, we accept reservations. Call us at (555) 123-4567 or book online.",
    "We serve contemporary American cuisine with seasonal ingredients.",
    "Yes, we have several vegetarian and vegan options on our menu.",
    "We're located at 123 Main Street, Downtown.",
    "Yes, we partner with major delivery services including DoorDash and Uber Eats.",
    "You can reach us at (555) 123-4567.",
    "Yes, we offer gluten-free bread and pasta options."
]

In [13]:
class AgentRouter:
    # Initialize the router with different agent instances
    def __init__(self, weather_agent, daily_dish_agent):
        self.weather_agent = weather_agent      # Agent responsible for weather-related queries
        self.daily_dish_agent = daily_dish_agent  # Agent responsible for restaurant/FAQ queries
        # Define keywords used to identify weather-related queries
        self.weather_keywords = [
            "weather", "temperature", "forecast",
            "rain", "sunny", "climate", "humidity",
            "hot", "cold", "warm"
        ]
    # Route the user query to the appropriate agent
    def route(self, query):
        # Convert the query to lowercase for case-insensitive matching
        query_lower = query.lower()
        # Check if the query contains any weather-related keywords
        for keyword in self.weather_keywords:
            if keyword in query_lower:
                return "weather"  # Route to the WeatherAgent
        # Default route if no weather keywords are found
        return "daily_dish"  # Route to the DailyDishAgent

In [ ]:
def answer(self, query):
    # Determine which agent should handle the query
    route = self.route(query)
    # Route to appropriate agent
    if route == "weather":
        return self.weather_agent.answer(query)
    else:
        return self.daily_dish_agent.answer(query)

In [14]:
# Initialize agents
weather_agent = WeatherAgent(api_key="your_api_key_here")
daily_dish_agent = DailyDishAgent(questions, answers)
# Create router
router = AgentRouter(weather_agent, daily_dish_agent)
# Handle queries
queries = [
    "What's the weather in Paris?",
    "Do you have vegetarian options?",
    "Is it going to rain in London?",
    "What are your opening hours?"
]
for query in queries:
    response = router.answer(query)
    print(f"Q: {query}")
    print(f"A: {response}\n")

NameError: name 'WeatherAgent' is not defined

In [ ]:
def route_with_confidence(self, query):
    query_lower = query.lower()
    scores = {"weather": 0, "daily_dish": 0}
    # Calculate weather score
    for keyword in self.weather_keywords:
        if keyword in query_lower:
            scores["weather"] += 1
    # Calculate restaurant score
    restaurant_keywords = [
        "menu", "food", "reservation", "hours",
        "restaurant", "dish", "eat", "dining"
    ]
    for keyword in restaurant_keywords:
        if keyword in query_lower:
            scores["daily_dish"] += 1
    # Return agent with highest score
    if scores["weather"] > scores["daily_dish"]:
        return "weather", scores["weather"]
    else:
        return "daily_dish", scores["daily_dish"]

In [15]:
def answer_with_fallback(self, query):
    try:
        route = self.route(query)
        if route == "weather":
            response = self.weather_agent.answer(query)
        else:
            response = self.daily_dish_agent.answer(query)
        # Check if response is valid
        if not response or response.startswith("Error"):
            return self.fallback_response(query)
        return response
    except Exception as e:
        return f"I encountered an issue: {str(e)}. Please try again."
def fallback_response(self, query):
    return (
        "I'm having trouble answering that right now. "
        "Please try rephrasing your question or contact us directly."
    )

In [16]:
import logging
from datetime import datetime
class MonitoredRouter(AgentRouter):
    # Initialize the monitored router with agent instances
    def __init__(self, weather_agent, daily_dish_agent):
        # Call the parent AgentRouter initializer
        super().__init__(weather_agent, daily_dish_agent)
        self.logs = []  # Store interaction logs in memory
        # Configure the logging system
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
    # Handle a user query with routing and monitoring
    def answer(self, query):
        # Record the start time of the request
        start_time = datetime.now()
        # Determine which agent should handle the query
        route = self.route(query)
        # Get the response from the appropriate agent via the parent class
        response = super().answer(query)
        # Create a structured log entry for this interaction
        log_entry = {
            "timestamp": start_time.isoformat(),          # Time the request was received
            "query": query,                               # User's input query
            "route": route,                               # Selected agent route
            "response": response,                         # Agent response
            "duration": (datetime.now() - start_time).total_seconds()  # Processing time in seconds
        }
        # Save the log entry for later analysis or auditing
        self.logs.append(log_entry)
        # Write a concise log message to the application logs
        self.logger.info(f"Query routed to {route}: {query[:50]}...")
        # Return the final response to the user
        return response
 

In [17]:
def test_agents():
    # Test data
    test_cases = [
        {
            "query": "What's the weather in Tokyo?",
            "expected_agent": "weather",
            "should_contain": ["Tokyo", "temperature"]
        },
        {
            "query": "Do you have vegan options?",
            "expected_agent": "daily_dish",
            "should_contain": ["vegetarian", "vegan"]
        }
    ]
    # Run tests
    for test in test_cases:
        response = router.answer(test["query"])
        route = router.route(test["query"])
        # Verify routing
        assert route == test["expected_agent"], \
            f"Expected {test['expected_agent']}, got {route}"
        # Verify response content
        for keyword in test["should_contain"]:
            assert keyword.lower() in response.lower(), \
                f"Response missing expected keyword: {keyword}"
        print(f"✓ Test passed: {test['query']}")


Key Concepts
Agent: An autonomous system that perceives its environment through sensors and acts upon that environment to achieve specific goals.

Routing: The process of directing user queries to the most appropriate specialized agent based on query content and context.

Memory: Storage mechanism that allows agents to maintain context and state across multiple interactions.

TF-IDF: Term Frequency-Inverse Document Frequency, a numerical statistic that reflects word importance in a document collection.

Cosine Similarity: A metric used to measure how similar two vectors are, commonly used for text similarity comparisons.

Semantic Matching: Finding the meaning-based similarity between texts, rather than exact keyword matches.